# 🫁 Medical Clinical RAG — Google Colab

**Scope:** Allergic Diseases in Adults/kids → **Asthma**

**Guideline:** *Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)* — NICE NG245, published 27 November 2024.

This notebook implements an end-to-end medical RAG prototype:

- PDF ingestion with page-aware extraction
- **Structure-aware clinical chunking** based on NICE sections and recommendation IDs
- Chunk metadata for section, recommendation, population, source and page
- Biomedical embeddings with MedCPT
- FAISS dense retrieval
- BM25 lexical retrieval
- Hybrid retrieval + score fusion
- MMR diversification
- Scope-aware clinical guardrails
- Optional LLM answer generation
- Citation and numerical consistency checks
- Asthma-specific retrieval test set

> ⚠️ This is a research/hackathon prototype, not a clinical decision-support system. It must not be used as a substitute for professional medical assessment.


In [2]:
# 1) Install dependencies
!pip -q install pymupdf faiss-cpu rank-bm25 sentence-transformers transformers accelerate openai pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.1 MB/s eta 0:00:00


In [3]:
# 2) Upload the medical PDF
from google.colab import files
from pathlib import Path

uploaded = files.upload()

PDF_PATH = next(
    (name for name in uploaded.keys() if name.lower().endswith(".pdf")),
    None
)

if PDF_PATH is None:
    raise ValueError("Please upload a PDF clinical guideline.")

print("Using:", PDF_PATH)

Saving asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf to asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf
Using: asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf


In [4]:
# 3) Imports and configuration
import re
import json
import math
import numpy as np
import pandas as pd
import fitz

from pathlib import Path
from rank_bm25 import BM25Okapi
from IPython.display import display, Markdown

DOCUMENT_ID = "NICE_NG245"
DOCUMENT_TITLE = "Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)"
SOURCE_AUTHORITY = "NICE / BTS / SIGN"
POPULATION = "Adults, young people and children (age-specific recommendations apply)"
CLINICAL_FIELD = "Allergic Diseases"
CONDITION = "Asthma"
SOURCE_URL = "https://www.nice.org.uk/guidance/ng245"

# Research-informed clinical chunk configuration.
# For structured clinical guidelines, logical boundaries are more important
# than blindly using a fixed character window.
MAX_CHUNK_TOKENS = 450
CHUNK_OVERLAP_TOKENS = 0

# Retrieval configuration
DENSE_TOP_K = 30
BM25_TOP_K = 30
FINAL_TOP_K = 6
MMR_LAMBDA = 0.70

print("Configuration loaded.")
print(f"Scope: {CLINICAL_FIELD} → {CONDITION}")
print(f"Guideline: {DOCUMENT_ID}")


Configuration loaded.
Scope: Allergic Diseases → Asthma
Guideline: NICE_NG245


## 1. PDF ingestion

The parser preserves page numbers and avoids destructive cleaning. Clinical numbers, units, negations, medication names, and recommendation wording are intentionally retained.

In [5]:
# 4) Extract page-aware text
def normalize_clinical_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = re.sub(r"-\n", "", text)       # repair line-break hyphenation
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

raw_pages = []

pdf = fitz.open(PDF_PATH)

for page_num, page in enumerate(pdf, start=1):
    text = normalize_clinical_text(page.get_text("text"))
    if text:
        raw_pages.append({
            "page": page_num,
            "text": text
        })

print(f"Pages extracted: {len(raw_pages)}")
print("First page preview:")
print(raw_pages[0]["text"][:1200])

Pages extracted: 64
First page preview:
Asthma: diagnosis, 
monitoring and chronic 
asthma management (BTS, 
NICE, SIGN) 
NICE guideline 
Published: 27 November 2024 
www.nice.org.uk/guidance/ng245 
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-andconditions#notice-of-rights).


## 2. Research-informed clinical chunking

The scope is a single NICE/BTS/SIGN asthma guideline. The chunker therefore uses the **document's own clinical structure** rather than treating the PDF as generic prose.

### Chunking design used here

1. **Section-aware:** detect headings such as `1.1 Initial clinical assessment`, `1.5 Monitoring asthma control`, and `1.7 Pharmacological management in people aged 12 and over`.
2. **Recommendation-aware:** detect numbered recommendations such as `1.1.1`, `1.2.1`, `1.7.3`, etc.
3. **Atomic recommendation principle:** keep a complete recommendation together whenever it fits the embedding limit.
4. **Long-recommendation fallback:** only long recommendations are split at sentence boundaries.
5. **Context enrichment:** each chunk stores the section, recommendation ID, page, population and source metadata. This avoids losing meaning when a retrieved chunk is viewed independently.
6. **No artificial overlap by default:** the guideline already provides strong semantic boundaries. Recent RAG research reports that overlap can increase indexing cost without reliably improving QA, while clinical RAG reviews recommend adapting chunk granularity to corpus structure.
7. **Size target:** 450 approximate tokens keeps chunks comfortably below the 512-token MedCPT input limit while leaving room for section context.

This is preferable here to a generic `RecursiveCharacterTextSplitter`, because NICE recommendations are meaningful clinical units and splitting them in the middle of a recommendation can separate conditions, thresholds, exceptions, or age groups.


In [6]:
# 5) Structure-aware asthma guideline chunking

# Examples:
#   1.1 Initial clinical assessment
#   1.2 Objective tests for diagnosing asthma...
#   1.7 Pharmacological management in people aged 12 and over
SECTION_RE = re.compile(
    r"(?m)^\s*(1\.\d+(?:\.\d+)?)\s+(.+?)\s*$"
)

# NICE recommendation IDs in this guideline are normally 3-part IDs:
# 1.1.1, 1.2.1, 1.7.10, etc.
RECOMMENDATION_RE = re.compile(
    r"(?m)(?<!\d)(\d+\.\d+\.\d+)\s+"
)

def token_count(text):
    """Lightweight token approximation used only for chunk sizing."""
    return max(1, len(re.findall(r"\S+", text)))

def split_by_sentences(text, max_tokens=MAX_CHUNK_TOKENS):
    """Split long clinical material without arbitrary character cuts."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    current = []

    for sentence in sentences:
        if not sentence:
            continue

        candidate = " ".join(current + [sentence])

        if current and token_count(candidate) > max_tokens:
            chunks.append(" ".join(current).strip())
            current = [sentence]
        else:
            current.append(sentence)

    if current:
        chunks.append(" ".join(current).strip())

    return [c for c in chunks if c]

def current_section_before(text, position):
    """Return the most recent NICE section heading before a position."""
    matches = list(SECTION_RE.finditer(text[:position]))
    if not matches:
        return {
            "section_id": "general",
            "section_title": "General guideline context"
        }

    m = matches[-1]
    return {
        "section_id": m.group(1),
        "section_title": m.group(2).strip()
    }

def infer_population(section_title, text):
    """Attach age/population metadata where the guideline explicitly signals it."""
    s = (section_title + " " + text).lower()

    if "children under 5" in s:
        return "Children under 5"
    if "children aged 5 to 11" in s:
        return "Children aged 5 to 11"
    if "children aged 5 to 16" in s:
        return "Children aged 5 to 16"
    if "aged 12 and over" in s:
        return "People aged 12 and over"
    if "adolescents" in s:
        return "Adolescents"
    if "pregnancy" in s or "breastfeeding" in s:
        return "People with asthma during pregnancy/breastfeeding"
    if "adults" in s:
        return "Adults"

    return POPULATION

def add_chunk(
    chunks,
    counter,
    text,
    page,
    section,
    recommendation_id,
    population
):
    text = text.strip()
    if not text:
        return counter

    counter += 1

    # Add a compact contextual prefix for embedding/retrieval while retaining
    # the original guideline wording in the metadata fields.
    context_prefix = (
        f"Condition: {CONDITION}. "
        f"Section {section['section_id']}: {section['section_title']}. "
        f"Population: {population}. "
        f"Recommendation: {recommendation_id}. "
    )

    indexed_text = context_prefix + text

    chunks.append({
        "chunk_id": f"{DOCUMENT_ID}_p{page}_{counter:04d}",
        "text": indexed_text,
        "source_text": text,
        "page": page,
        "section_id": section["section_id"],
        "section": section["section_title"],
        "recommendation_id": recommendation_id,
        "document_id": DOCUMENT_ID,
        "document_type": "clinical_guideline",
        "clinical_field": CLINICAL_FIELD,
        "condition": CONDITION,
        "source_authority": SOURCE_AUTHORITY,
        "population": population,
        "title": DOCUMENT_TITLE,
        "source_url": SOURCE_URL
    })

    return counter

def make_chunks(pages):
    chunks = []
    chunk_counter = 0

    for page in pages:
        text = page["text"]

        # Find recommendation boundaries. A recommendation is treated as the
        # primary retrieval unit because it is a clinically meaningful statement.
        matches = list(RECOMMENDATION_RE.finditer(text))

        if not matches:
            # Pages such as contents/overview/definitions/rationale may not
            # contain numbered recommendations. Keep their semantic sections.
            section = current_section_before(text, len(text))
            population = infer_population(section["section_title"], text)

            for part in split_by_sentences(text):
                chunk_counter = add_chunk(
                    chunks,
                    chunk_counter,
                    part,
                    page["page"],
                    section,
                    "general",
                    population
                )
            continue

        # Material before the first recommendation.
        if matches[0].start() > 0:
            pre = text[:matches[0].start()].strip()

            if pre:
                section = current_section_before(text, matches[0].start())
                population = infer_population(section["section_title"], pre)

                for part in split_by_sentences(pre):
                    chunk_counter = add_chunk(
                        chunks,
                        chunk_counter,
                        part,
                        page["page"],
                        section,
                        "general",
                        population
                    )

        # Recommendation-level chunks.
        for i, match in enumerate(matches):
            start = match.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

            rec_id = match.group(1)
            rec_text = text[start:end].strip()

            section = current_section_before(text, start)
            population = infer_population(section["section_title"], rec_text)

            if token_count(rec_text) <= MAX_CHUNK_TOKENS:
                chunk_counter = add_chunk(
                    chunks,
                    chunk_counter,
                    rec_text,
                    page["page"],
                    section,
                    rec_id,
                    population
                )
            else:
                # Only long recommendations are sentence-split.
                parts = split_by_sentences(rec_text)

                for part in parts:
                    chunk_counter = add_chunk(
                        chunks,
                        chunk_counter,
                        part,
                        page["page"],
                        section,
                        rec_id,
                        population
                    )

    return chunks

chunks = make_chunks(raw_pages)

chunk_df = pd.DataFrame(chunks)

print("Chunks:", len(chunks))
print("Mean approximate tokens:", round(chunk_df["text"].map(token_count).mean(), 1))
print("Max approximate tokens:", chunk_df["text"].map(token_count).max())
print("Recommendation chunks:", (chunk_df["recommendation_id"] != "general").sum())

display(
    chunk_df.head(12)[[
        "chunk_id",
        "page",
        "section",
        "recommendation_id",
        "population",
        "text"
    ]]
)


Chunks: 175
Mean approximate tokens: 117.2
Max approximate tokens: 465
Recommendation chunks: 111


,chunk_id,page,section,recommendation_id,population,text
0,NICE_NG245_p1_0001,1,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
1,NICE_NG245_p2_0002,2,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
2,NICE_NG245_p3_0003,3,Organisation and delivery of care ...............,general,Children under 5,Condition: Asthma. Section 1.16: Organisation ...
3,NICE_NG245_p4_0004,4,General guideline context,general,Children under 5,Condition: Asthma. Section general: General gu...
4,NICE_NG245_p5_0005,5,General guideline context,general,Adults,Condition: Asthma. Section general: General gu...
5,NICE_NG245_p6_0006,6,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
6,NICE_NG245_p7_0007,7,General guideline context,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section general: General gu...
7,NICE_NG245_p8_0008,8,Initial clinical assessment,general,"Adults, young people and children (age-specifi...",Condition: Asthma. Section 1.1: Initial clinic...
8,NICE_NG245_p8_0009,8,Initial clinical assessment,1.1.1,Adults,Condition: Asthma. Section 1.1: Initial clinic...
9,NICE_NG245_p8_0010,8,Obtain a structured clinical history in people...,1.1.2,"Adults, young people and children (age-specifi...",Condition: Asthma. Section 1.1.1: Obtain a str...


## 🔬 Why this chunking strategy?

The chunking choice is based on recent RAG and clinical-RAG research:

- A 2026 systematic chunking analysis found sentence-based chunking to be cost-effective and reported that overlap did not provide a measurable benefit in its evaluated setting.
- A systematic review of biomedical RAG recommends optimizing chunk granularity according to the structure of the clinical corpus; it specifically notes that fixed-length chunking can fragment related information and recommends combining structural/sparse metadata with dense retrieval.
- A 2025 clinical decision-support comparison found adaptive/logical-boundary chunking improved retrieval and clinical-answer quality over a fixed baseline.

For this NICE guideline, the most defensible implementation is therefore **recommendation-aware + section-aware chunking**, with metadata preservation and no forced overlap. The notebook can later compare this against recursive/fixed chunking using the same retrieval and evaluation pipeline.


## 3. Medical embeddings

MedCPT is used because this is a biomedical/clinical retrieval task. The article encoder indexes guideline chunks and the query encoder embeds user questions.

The chunking stage keeps each retrieval unit clinically coherent before embedding it.


In [7]:
# 6) Load biomedical embedding model
from transformers import AutoTokenizer, AutoModel
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
ARTICLE_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"

query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_model = AutoModel.from_pretrained(QUERY_MODEL_NAME).to(DEVICE)

article_tokenizer = AutoTokenizer.from_pretrained(ARTICLE_MODEL_NAME)
article_model = AutoModel.from_pretrained(ARTICLE_MODEL_NAME).to(DEVICE)

query_model.eval()
article_model.eval()

def mean_pool(output, attention_mask):
    token_embeddings = output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(texts, tokenizer, model, batch_size=16):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = model(**inputs)
        emb = mean_pool(outputs, inputs["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

def encode_query(query):
    return encode_texts(
        [query],
        query_tokenizer,
        query_model,
        batch_size=1
    )[0]

texts = [c["text"] for c in chunks]
embeddings = encode_texts(texts, article_tokenizer, article_model)

print("Embedding matrix:", embeddings.shape)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding matrix: (175, 768)


## 4. FAISS + BM25 hybrid retrieval

In [8]:
# 7) Build dense FAISS index and BM25 index
import faiss

dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings.astype("float32"))

bm25_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(bm25_corpus)

print("FAISS vectors:", faiss_index.ntotal)
print("BM25 documents:", len(bm25_corpus))

FAISS vectors: 175
BM25 documents: 175


In [9]:
# 8) Hybrid retrieval + metadata filtering + MMR
# Metadata filters are asthma-specific and preserve age/section provenance.
def minmax(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-9:
        return np.ones_like(values)
    return (values - lo) / (hi - lo)

def cosine_sim(a, b):
    return float(np.dot(a, b))

def mmr_select(query_embedding, candidate_indices, relevance_scores, k=FINAL_TOP_K, lambda_mult=MMR_LAMBDA):
    selected = []
    remaining = list(candidate_indices)

    while remaining and len(selected) < k:
        best_idx = None
        best_value = -1e9

        for idx in remaining:
            relevance = relevance_scores[idx]

            if not selected:
                diversity_penalty = 0.0
            else:
                diversity_penalty = max(
                    cosine_sim(embeddings[idx], embeddings[j])
                    for j in selected
                )

            value = (
                lambda_mult * relevance
                - (1 - lambda_mult) * diversity_penalty
            )

            if value > best_value:
                best_value = value
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return selected

def hybrid_search(
    query,
    top_k=FINAL_TOP_K,
    condition=None,
    recommendation_id=None,
    population=None
):
    # Dense candidates
    q = encode_query(query).astype("float32").reshape(1, -1)
    dense_scores, dense_ids = faiss_index.search(
        q, min(DENSE_TOP_K, len(chunks))
    )

    dense_scores = dense_scores[0]
    dense_ids = dense_ids[0]

    # BM25 candidates
    bm_scores_all = bm25.get_scores(query.lower().split())
    bm_ids = np.argsort(bm_scores_all)[::-1][:min(BM25_TOP_K, len(chunks))]

    candidate_ids = set(int(i) for i in dense_ids if i >= 0)
    candidate_ids.update(int(i) for i in bm_ids)

    # Optional metadata/content filter
    filtered = []
    for idx in candidate_ids:
        item = chunks[idx]

        if recommendation_id is not None:
            if item["recommendation_id"] != recommendation_id:
                continue

        if condition is not None:
            if condition.lower() not in item["condition"].lower():
                continue

        if population is not None:
            if population.lower() not in item["population"].lower():
                continue

        filtered.append(idx)

    if not filtered:
        return []

    dense_map = {int(i): float(s) for i, s in zip(dense_ids, dense_scores)}
    bm_map = {int(i): float(bm_scores_all[i]) for i in bm_ids}

    dense_raw = np.array([dense_map.get(i, 0.0) for i in filtered])
    bm_raw = np.array([bm_map.get(i, 0.0) for i in filtered])

    dense_norm = minmax(dense_raw)
    bm_norm = minmax(bm_raw)

    # Weighted fusion: dense 65%, lexical 35%
    fused = 0.65 * dense_norm + 0.35 * bm_norm
    score_map = {idx: float(score) for idx, score in zip(filtered, fused)}

    selected = mmr_select(
        q[0],
        filtered,
        score_map,
        k=top_k
    )

    results = []
    for idx in selected:
        item = dict(chunks[idx])
        item["dense_score"] = dense_map.get(idx, 0.0)
        item["bm25_score"] = bm_map.get(idx, 0.0)
        item["hybrid_score"] = score_map[idx]
        results.append(item)

    return results

def show_results(results):
    rows = []
    for r in results:
        rows.append({
            "page": r["page"],
            "recommendation": r["recommendation_id"],
            "hybrid_score": round(r["hybrid_score"], 4),
            "text": r["text"][:500]
        })
    display(pd.DataFrame(rows))

# Test retrieval
query = "What are the red flags that should prompt further investigation or referral for headache?"
results = hybrid_search(query)
show_results(results)

,page,recommendation,hybrid_score,text
0,58,general,0.9566,Condition: Asthma. Section general: General gu...
1,60,general,0.8274,Condition: Asthma. Section general: General gu...
2,46,general,0.8194,Condition: Asthma. Section general: General gu...
3,25,1.9.3,0.8103,Condition: Asthma. Section general: General gu...
4,8,general,0.6406,Condition: Asthma. Section 1.1: Initial clinic...
5,54,general,0.6443,Condition: Asthma. Section general: General gu...


## 5. Clinical safety guardrails

The system should not answer outside the current knowledge-base scope. It should also detect obvious red-flag language and avoid presenting the RAG output as a diagnosis or prescription.

In [10]:
# 9) Asthma-specific safety and scope gate

RED_FLAG_PATTERNS = [
    "acute asthma attack",
    "severe asthma attack",
    "difficulty breathing",
    "struggling to breathe",
    "cannot speak",
    "hospital admission",
    "emergency department",
    "acute exacerbation"
]

IN_SCOPE_TERMS = [
    "asthma",
    "wheeze",
    "wheezing",
    "breathlessness",
    "chest tightness",
    "cough",
    "fev1",
    "feno",
    "eosinophil",
    "peak expiratory flow",
    "pef",
    "spirometry",
    "bronchodilator reversibility",
    "inhaled corticosteroid",
    "ics",
    "formoterol",
    "mart",
    "air therapy",
    "saba",
    "laba",
    "lama",
    "ltra",
    "inhaler",
    "inhaler technique",
    "asthma control",
    "asthma action plan"
]

def safety_check(query):
    q = query.lower()

    red_flags = [p for p in RED_FLAG_PATTERNS if p in q]
    in_scope = any(term in q for term in IN_SCOPE_TERMS)

    return {
        "red_flags_detected": red_flags,
        "in_scope": in_scope,
        "scope": f"{CLINICAL_FIELD} → {CONDITION}",
        "population": POPULATION
    }

def build_context(results):
    blocks = []

    for i, r in enumerate(results, start=1):
        citation = (
            f"[NICE NG245 | Section {r['section_id']} | "
            f"Recommendation {r['recommendation_id']} | "
            f"Page {r['page']} | Population: {r['population']}]"
        )

        blocks.append(
            f"Evidence {i} {citation}\n{r['source_text']}"
        )

    return "\n\n---\n\n".join(blocks)

def citation_for(r):
    return (
        f"[NICE NG245, Section {r['section_id']}, "
        f"Recommendation {r['recommendation_id']}, p.{r['page']}]"
    )


## 6. Optional LLM answer generation

This cell uses an OpenAI API key entered privately in Colab. The LLM is instructed to answer only from retrieved evidence and to cite each clinical claim.

If you do not have an API key, the notebook still works as a retrieval system and displays the evidence chunks.

In [11]:
# 10) Optional LLM generation
from getpass import getpass

USE_LLM = False

if USE_LLM:
    from openai import OpenAI

    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY: ")
    client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = '''
You are a clinical evidence retrieval assistant for an asthma guideline.

Scope:
- Clinical field: Allergic Diseases
- Condition: Asthma
- Source: Asthma: diagnosis, monitoring and chronic asthma management
  (BTS, NICE, SIGN), NICE NG245
- Population: adults, young people and children, with age-specific
  recommendations preserved.

Use ONLY the supplied retrieved evidence.
Do not invent clinical facts, recommendations, dosages, contraindications,
age groups, or citations.
Do not diagnose a patient.
Do not prescribe individualized treatment.
Preserve all numerical values exactly.
Preserve negative recommendations such as "do not".
If the evidence is insufficient, say that the available source does not
provide enough information.

Every clinically relevant claim must include a citation in this format:
[NICE NG245, Section X.X, Recommendation X.X.X, p.X]

This is guideline information, not individualized medical advice.
'''

def answer_query(query, use_llm=USE_LLM):
    safety = safety_check(query)

    if not safety["in_scope"]:
        return {
            "answer": (
                "The current knowledge base is scoped to asthma guidance "
                "within allergic diseases for adults, young people and "
                "children. The available source does not provide enough "
                "information to answer this question reliably."
            ),
            "sources": []
        }

    results = hybrid_search(query, condition=CONDITION)

    if not results:
        return {
            "answer": "No sufficiently relevant asthma guideline evidence was retrieved.",
            "sources": []
        }

    context = build_context(results)

    # Safe fallback: evidence-only response
    if not use_llm:
        return {
            "answer": (
                "LLM generation is disabled. The following NICE NG245 "
                "guideline evidence was retrieved for the query:"

 + context
            ),
            "sources": results
        }

    user_prompt = f'''
Clinical evidence:
{context}

User question:
{query}

Answer only from the evidence above. Include precise citations.
'''

    response = client.chat.completions.create(
        model="gpt-5.6",
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": results
    }

# Example
response = answer_query(
    "What objective tests are used to diagnose asthma in adults?",
    use_llm=False
)

print(response["answer"])


LLM generation is disabled. The following NICE NG245 guideline evidence was retrieved for the query:Evidence 1 [NICE NG245 | Section 1.1.6 | Recommendation 1.1.7 | Page 9 | Population: Children aged 5 to 16]
1.1.7 
Be aware that the results of spirometry and FeNO tests may be affected in people 
who have been treated with inhaled corticosteroids (the test results are more 
likely to be normal). [NICE 2017] 
1.2 Objective tests for diagnosing asthma in adults, 
young people and children aged 5 to 16 with a 
history suggestive of asthma 
Adults 
See also algorithm A for a summary of objective tests for diagnosing asthma in adults and 
young people (aged over 16 years) with a history suggesting asthma. 
Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)
(NG245)
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-andconditions#notice-of-rights).
Page 9 of
64

---

Evidence 2 [NICE NG245 | Section 1.2.3 | Recommendation 1.2

## 7. Citation + numerical consistency checks

These checks are deliberately conservative. In a medical RAG system, a wrong citation or changed number should be treated as a failure rather than silently corrected.

In [12]:
# 11) Lightweight validation
def extract_numbers(text):
    return re.findall(
        r"(?<!\w)(?:\d+(?:\.\d+)?)(?:\s*[-–]\s*\d+(?:\.\d+)?)?",
        text
    )

def validate_citations(answer, sources):
    valid = {
        citation_for(r)
        for r in sources
    }

    found = re.findall(
        r"\[NICE NG245, Section [^\]]+, Recommendation [^\]]+, p\.\d+\]",
        answer
    )

    invalid = [c for c in found if c not in valid]

    return {
        "citations_found": found,
        "invalid_citations": invalid,
        "citation_valid": len(invalid) == 0
    }

def validate_numbers(answer, sources):
    source_text = " ".join(r["text"] for r in sources)
    answer_numbers = set(extract_numbers(answer))
    source_numbers = set(extract_numbers(source_text))

    unsupported = sorted(answer_numbers - source_numbers)

    return {
        "answer_numbers": sorted(answer_numbers),
        "unsupported_numbers": unsupported,
        "numeric_check": len(unsupported) == 0
    }

def validate_response(answer, sources):
    citation_result = validate_citations(answer, sources)
    number_result = validate_numbers(answer, sources)

    return {
        **citation_result,
        **number_result,
        "safe_to_display": (
            citation_result["citation_valid"]
            and number_result["numeric_check"]
        )
    }

# Example validation
if response["sources"]:
    validation = validate_response(
        response["answer"],
        response["sources"]
    )
    print(json.dumps(validation, indent=2))

{
  "citations_found": [],
  "invalid_citations": [],
  "citation_valid": true,
  "answer_numbers": [
    "1",
    "1.1",
    "1.2",
    "1.3",
    "1.4",
    "10",
    "12",
    "13",
    "16",
    "2",
    "2017",
    "2024",
    "2026",
    "3",
    "4",
    "41",
    "46",
    "5",
    "6",
    "64",
    "7",
    "9"
  ],
  "unsupported_numbers": [
    "10",
    "13",
    "41",
    "46"
  ],
  "numeric_check": false,
  "safe_to_display": false
}


## 8. Asthma retrieval test set

These tests are designed from the supplied NICE NG245 asthma guideline. They cover:
- initial clinical assessment
- objective diagnosis
- monitoring asthma control
- pharmacological management
- self-management
- risk-stratified care

They evaluate retrieval behavior; they do not claim clinical performance without a labeled benchmark.


In [13]:
# 12) Asthma retrieval evaluation examples
TEST_QUERIES = [
    "What clinical history should be obtained when asthma is suspected?",
    "What objective tests can diagnose asthma in adults?",
    "What are the FeNO and bronchodilator reversibility thresholds for diagnosing asthma?",
    "How should asthma control be monitored at every review?",
    "What is the initial treatment for newly diagnosed asthma in people aged 12 and over?",
    "What is the initial treatment for children aged 5 to 11 with newly diagnosed asthma?",
    "What should be checked before escalating asthma medicines?",
    "What should be included in an asthma self-management programme?",
    "Which people with asthma are at increased risk of poor outcomes?"
]

evaluation_rows = []

for q in TEST_QUERIES:
    retrieved = hybrid_search(
        q,
        top_k=6,
        condition=CONDITION
    )

    evaluation_rows.append({
        "query": q,
        "retrieved_chunks": len(retrieved),
        "top_recommendations": ", ".join(
            sorted(set(r["recommendation_id"] for r in retrieved))
        ),
        "top_sections": ", ".join(
            sorted(set(r["section_id"] for r in retrieved))
        ),
        "top_pages": ", ".join(str(r["page"]) for r in retrieved[:3])
    })

display(pd.DataFrame(evaluation_rows))


,query,retrieved_chunks,top_recommendations,top_sections,top_pages
0,What clinical history should be obtained when ...,6,"1.1.1, 1.1.2, 1.2.9, 1.3.2, 1.6.3, general","1.1, 1.1.1, 1.3.1, general","8, 41, 49"
1,What objective tests can diagnose asthma in ad...,6,"1.1.7, 1.2.4, 1.2.9, 1.3.1, 1.3.3, general","1.1.6, 1.2.3, 1.3, 1.3.2, general","10, 13, 9"
2,What are the FeNO and bronchodilator reversibi...,6,"1.2.2, general","1.2.1, general","42, 10, 46"
3,How should asthma control be monitored at ever...,6,"1.10.3, 1.14.5, 1.5.1, 1.5.2, 1.6.1, 1.6.7","1.10.2, 1.5, 1.5.1, 1.6.6, general","14, 14, 17"
4,What is the initial treatment for newly diagno...,6,"1.7.6, general","1.7, general","19, 18, 50"
5,What is the initial treatment for children age...,6,"1.3.1, 1.7.6, general","1.3, 1.9, general","55, 22, 58"
6,What should be checked before escalating asthm...,6,"1.14.4, 1.16.1, 1.6.1, 1.6.3, 1.6.7, general","1.14.3, 1.16, 1.6.6, general","54, 17, 49"
7,What should be included in an asthma self-mana...,6,"1.14.4, 1.14.5, 1.16.2, general","1.14.3, 1.14.4, 1.16.1, general","29, 40, 58"
8,Which people with asthma are at increased risk...,6,"1.1.4, 1.10.1, 1.14.5, 1.15.1, 1.16.1, 1.6.3","1.10, 1.15, 1.16, general","30, 49, 9"


## 9. Save the local vector/RAG artifacts

For a hackathon demo, this lets you avoid rebuilding embeddings every time the notebook is reopened. The files remain local to the current Colab runtime unless you copy them to Google Drive.

In [14]:
# 13) Save indexes and metadata
import pickle

faiss.write_index(faiss_index, "medical_rag.faiss")

with open("medical_rag_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

np.save("medical_rag_embeddings.npy", embeddings)

print("Saved:")
print("- medical_rag.faiss")
print("- medical_rag_chunks.pkl")
print("- medical_rag_embeddings.npy")

Saved:
- medical_rag.faiss
- medical_rag_chunks.pkl
- medical_rag_embeddings.npy


# ✅ Asthma RAG demo

Change the question below and run the cell.

The pipeline is:

**Query → Asthma Scope/Safety Gate → MedCPT Dense Search + BM25 → Hybrid Fusion → MMR → NICE NG245 Evidence → Optional LLM → Citation/Numeric Validation**


In [15]:
# 14) Interactive demo

user_query = "What objective tests are recommended to diagnose asthma in adults?"

safety = safety_check(user_query)
print("\nSafety:", safety)

if safety["red_flags_detected"]:
    print(
        "\n⚠️ Red-flag language detected. "
        "The query should be handled with appropriate clinical urgency."
    )

demo = answer_query(user_query, use_llm=USE_LLM)

print("\nANSWER / EVIDENCE\n")
print(demo["answer"])

if demo["sources"]:
    print("\nSOURCES\n")
    for r in demo["sources"]:
        print(citation_for(r))


Safety: {'red_flags_detected': [], 'in_scope': True, 'scope': 'Allergic Diseases → Asthma', 'population': 'Adults, young people and children (age-specific recommendations apply)'}

ANSWER / EVIDENCE

LLM generation is disabled. The following NICE NG245 guideline evidence was retrieved for the query:Evidence 1 [NICE NG245 | Section 1.3 | Recommendation 1.3.1 | Page 13 | Population: Children under 5]
1.3.1 
For children under 5 with suspected asthma, treat with inhaled corticosteroids in 
line with the recommendations on medicines for initial management in children 
under 5, and review the child on a regular basis. If they still have symptoms when 
they reach 5 years, attempt objective tests (see the section on objective tests for 
diagnosing asthma in adults, young people and children aged 5 to 16). [NICE 
2017]

---

Evidence 2 [NICE NG245 | Section general | Recommendation general | Page 46 | Population: Children under 5]
In view of the difficulty in diagnosing asthma in this age gro

In [16]:
# Test Questions 1-10: Answerable

questions = [
    "What FeNO level supports a diagnosis of asthma in adults?",

    "What FEV1 increase is required to diagnose asthma using bronchodilator reversibility in adults?",

    "How often should peak expiratory flow be measured when spirometry is unavailable or delayed?",

    "What PEF variability supports a diagnosis of asthma in adults?",

    "When should a person with suspected asthma be referred for a bronchial challenge test?",

    "What FeNO level supports a diagnosis of asthma in children aged 5 to 16?",

    "What should be done if a child under 5 cannot perform objective tests at age 5?",

    "What factors should be checked when monitoring asthma control at every review?",

    "Which validated symptom questionnaires can be considered during an asthma review?",

    "Should regular PEF monitoring be used routinely to assess asthma control?"
]

for i, user_query in enumerate(questions, 1):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print("=" * 80)
    print(user_query)

    safety = safety_check(user_query)
    print("\nSafety:", safety)

    if safety["red_flags_detected"]:
        print(
            "\n⚠️ Red-flag language detected. "
            "The query should be handled with appropriate clinical urgency."
        )

    demo = answer_query(user_query, use_llm=USE_LLM)

    print("\nANSWER / EVIDENCE\n")
    print(demo["answer"])

    if demo["sources"]:
        print("\nSOURCES\n")
        for r in demo["sources"]:
            print(citation_for(r))


QUESTION 1
What FeNO level supports a diagnosis of asthma in adults?

Safety: {'red_flags_detected': [], 'in_scope': True, 'scope': 'Allergic Diseases → Asthma', 'population': 'Adults, young people and children (age-specific recommendations apply)'}

ANSWER / EVIDENCE

LLM generation is disabled. The following NICE NG245 guideline evidence was retrieved for the query:Evidence 1 [NICE NG245 | Section general | Recommendation general | Page 48 | Population: Adults]
circumstances. FeNO 
The evidence showed that, in both adults and children, regular FeNO monitoring led to a 
reduction in the number of asthma exacerbations. In children there was also a significant 
improvement in lung function. In adults, the reduction in exacerbations was achieved 
alongside an overall reduction in the dosage of maintenance ICS therapy. This was not the 
case in children, but the studies in this age group were more likely to be conducted in 
secondary or tertiary care, so it is likely that they had a high

In [21]:
# Test Questions 11-15: Paraphrased

questions = [
    "For an adult suspected of having asthma, what FeNO measurement would be high enough to support the diagnosis?",

    "If spirometry is delayed, what alternative measurement can be taken over a two-week period to help confirm asthma?",

    "If the initial tests do not confirm asthma but clinical suspicion remains, what further investigation can be considered?",

    "During a routine asthma review, what information about reliever medication use should be assessed?",

    "What types of questionnaires can help assess asthma symptoms during a clinical review?"
]

for i, user_query in enumerate(questions, 11):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print("=" * 80)
    print(user_query)

    safety = safety_check(user_query)
    print("\nSafety:", safety)

    if safety["red_flags_detected"]:
        print(
            "\n⚠️ Red-flag language detected. "
            "The query should be handled with appropriate clinical urgency."
        )

    demo = answer_query(user_query, use_llm=USE_LLM)

    print("\nANSWER / EVIDENCE\n")
    print(demo["answer"])

    if demo["sources"]:
        print("\nSOURCES\n")
        for r in demo["sources"]:
            print(citation_for(r))


QUESTION 11
For an adult suspected of having asthma, what FeNO measurement would be high enough to support the diagnosis?

Safety: {'red_flags_detected': [], 'in_scope': True, 'scope': 'Allergic Diseases → Asthma', 'population': 'Adults, young people and children (age-specific recommendations apply)'}

ANSWER / EVIDENCE

LLM generation is disabled. The following NICE NG245 guideline evidence was retrieved for the query:Evidence 1 [NICE NG245 | Section general | Recommendation general | Page 42 | Population: Adults]
similar information to a preceding one. Practical aspects were taken into account using the 
committee's knowledge and experience. These included the availability of the tests, which 
varies considerably (in particular, bronchial challenge testing is not available in primary 
care and not readily available in secondary care), the ability of people to perform the tests, 
and the acceptability of the tests to the person, which is particularly relevant in younger 
children. Th

In [23]:
# Test Questions 16-20: Questions outside the provided guideline

questions = [
    "What is the prevalence of asthma in Egypt?",

    "What is the average annual cost of asthma treatment per patient?",

    "Which asthma inhaler brand is the most prescribed worldwide?",

    "What percentage of asthma patients in the United States are hospitalized each year?",

    "What is the average age of asthma diagnosis worldwide?"
]

for i, user_query in enumerate(questions, 16):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print("=" * 80)
    print(user_query)

    safety = safety_check(user_query)
    print("\nSafety:", safety)

    if safety["red_flags_detected"]:
        print(
            "\n⚠️ Red-flag language detected. "
            "The query should be handled with appropriate clinical urgency."
        )

    demo = answer_query(user_query, use_llm=USE_LLM)

    print("\nANSWER / EVIDENCE\n")
    print(demo["answer"])

    if demo["sources"]:
        print("\nSOURCES\n")
        for r in demo["sources"]:
            print(citation_for(r))


QUESTION 16
What is the prevalence of asthma in Egypt?

Safety: {'red_flags_detected': [], 'in_scope': True, 'scope': 'Allergic Diseases → Asthma', 'population': 'Adults, young people and children (age-specific recommendations apply)'}

ANSWER / EVIDENCE

LLM generation is disabled. The following NICE NG245 guideline evidence was retrieved for the query:Evidence 1 [NICE NG245 | Section general | Recommendation general | Page 61 | Population: Adults, young people and children (age-specific recommendations apply)]
Context 
The NICE guideline on asthma was published in 2017 and BTS/SIGN last updated their 
asthma guideline in 2019. The guidelines overlap in the clinical areas included, and 
healthcare practitioners in the UK have been using both sets of guidance. However, these guidelines differ in their approach to diagnosis. Concern has been raised 
about the recommendations to use fractional exhaled nitric oxide (FeNO) measurement 
and spirometry more widely, contained in NICE guidanc